## 1. Initialize Project Environment
Import libraries for network visualization and hub gene identification.

In [ ]:
from __future__ import annotations

import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Set

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")


def locate_repo_root() -> Path:
    """Find the repository root by looking for data folder."""
    here = Path().resolve()
    for base in [here, *here.parents]:
        if (base / "data").exists():
            return base
    raise FileNotFoundError("Could not locate repository root")


REPO_ROOT = locate_repo_root()
ARTIFACTS = REPO_ROOT / "labs/07_network_viz/assignments/artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

logging.info("Repo root: %s", REPO_ROOT)
logging.info("Artifacts directory: %s", ARTIFACTS)

## 2. Define Configuration Parameters
Centralize visualization settings: layout, colors, hub criteria.

In [ ]:
@dataclass
class VizConfig:
    handle: str
    layout: str = "spring"  # "spring", "kamada_kawai", "circular"
    seed: int = 42
    top_k_hubs: int = 10
    node_base_size: int = 60
    hub_size_multiplier: float = 3.0
    edge_alpha: float = 0.15
    figsize: tuple = (14, 12)
    dpi: int = 200
    export_dir: Path = None

    def __post_init__(self):
        if self.export_dir is None:
            self.export_dir = ARTIFACTS

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["export_dir"] = str(info["export_dir"])
        info["figsize"] = str(info["figsize"])
        return info


CONFIG = VizConfig(handle="AndreiCod")
CONFIG.describe()

In [ ]:
# Load data from Task 2
modules_path = ARTIFACTS / f"modules_tp53_{CONFIG.handle}.csv"
adj_path = ARTIFACTS / "task2_adjacency_matrix.csv"

if not modules_path.exists() or not adj_path.exists():
    raise FileNotFoundError(
        "Task 2 outputs not found. Run Task2_network_modules.ipynb first."
    )

# Load modules
modules_df = pd.read_csv(modules_path)
gene2module = dict(zip(modules_df["Gene"], modules_df["Module"]))
logging.info("Loaded %d gene-module mappings", len(gene2module))

# Load adjacency matrix
adj_matrix = pd.read_csv(adj_path, index_col=0)
logging.info(
    "Loaded adjacency matrix: %d × %d", adj_matrix.shape[0], adj_matrix.shape[1]
)

In [ ]:
# Build graph from adjacency
G = nx.from_pandas_adjacency(adj_matrix)
isolates = list(nx.isolates(G))
G.remove_nodes_from(isolates)
logging.info("Graph: %d nodes, %d edges", G.number_of_nodes(), G.number_of_edges())

## 3. Implement Core Functionality
Compute hub genes, assign colors by module, and create the network visualization.

In [ ]:
def compute_hub_genes(
    G: nx.Graph, gene2module: Dict[str, int], top_k: int
) -> pd.DataFrame:
    """Identify top-k hub genes by degree centrality.

    Args:
        G: NetworkX graph
        gene2module: Mapping of gene to module ID
        top_k: Number of top hubs to return

    Returns:
        DataFrame with Gene, Degree, and DegreeCentrality columns
    """
    degrees = dict(G.degree())
    centrality = nx.degree_centrality(G)

    hub_data = []
    for gene in degrees:
        hub_data.append(
            {
                "Gene": gene,
                "Degree": degrees[gene],
                "DegreeCentrality": centrality[gene],
                "Module": gene2module.get(gene, -1),
            }
        )

    df = pd.DataFrame(hub_data)
    df = df.sort_values("Degree", ascending=False).head(top_k).reset_index(drop=True)
    return df


def get_node_colors(G: nx.Graph, gene2module: Dict[str, int]) -> List:
    """Assign colors to nodes based on module membership."""
    cmap = plt.get_cmap("tab10")
    colors = []
    for node in G.nodes():
        module = gene2module.get(node, 0)
        colors.append(cmap(module % 10))
    return colors


def get_node_sizes(
    G: nx.Graph, hub_genes: Set[str], base_size: int, multiplier: float
) -> List[int]:
    """Compute node sizes, larger for hub genes."""
    sizes = []
    for node in G.nodes():
        if node in hub_genes:
            sizes.append(int(base_size * multiplier))
        else:
            sizes.append(base_size)
    return sizes

In [ ]:
# Compute hub genes
hubs_df = compute_hub_genes(G, gene2module, CONFIG.top_k_hubs)
hub_genes = set(hubs_df["Gene"])

print(f"Top {CONFIG.top_k_hubs} Hub Genes:")
print(hubs_df.to_string(index=False))

In [ ]:
# Prepare visualization parameters
node_colors = get_node_colors(G, gene2module)
node_sizes = get_node_sizes(
    G, hub_genes, CONFIG.node_base_size, CONFIG.hub_size_multiplier
)

# Compute layout
if CONFIG.layout == "spring":
    pos = nx.spring_layout(G, seed=CONFIG.seed, k=1.5)
elif CONFIG.layout == "kamada_kawai":
    pos = nx.kamada_kawai_layout(G)
elif CONFIG.layout == "circular":
    pos = nx.circular_layout(G)
else:
    pos = nx.spring_layout(G, seed=CONFIG.seed)

logging.info("Layout computed: %s", CONFIG.layout)

In [ ]:
# Create visualization
fig, ax = plt.subplots(figsize=CONFIG.figsize)

# Draw edges
nx.draw_networkx_edges(G, pos, alpha=CONFIG.edge_alpha, edge_color="gray", ax=ax)

# Draw nodes (colored by module)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, ax=ax)

# Draw labels only for hub genes
hub_labels = {n: n for n in hub_genes if n in G.nodes()}
nx.draw_networkx_labels(
    G, pos, labels=hub_labels, font_size=9, font_weight="bold", ax=ax
)

# Add title with network statistics
num_modules = len(set(gene2module.values()))
ax.set_title(
    f"Gene Co-Expression Network — TP53 Associated\n"
    f"{G.number_of_nodes()} genes, {G.number_of_edges()} edges, {num_modules} modules",
    fontsize=14,
)

# Create legend for modules
cmap = plt.get_cmap("tab10")
unique_modules = sorted(set(gene2module.values()))
legend_handles = [
    plt.Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=cmap(m % 10),
        markersize=10,
        label=f"Module {m}",
    )
    for m in unique_modules
]
ax.legend(handles=legend_handles, loc="upper left", title="Modules")

ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Validate with Unit Tests
Ensure hub detection and coloring work correctly.

In [ ]:
def test_hub_detection():
    """Test hub gene detection on a simple graph."""
    test_G = nx.Graph()
    # Star graph: central node connected to all others
    test_G.add_edges_from([("hub", f"leaf_{i}") for i in range(5)])
    test_gene2mod = {"hub": 0, **{f"leaf_{i}": 1 for i in range(5)}}

    hubs = compute_hub_genes(test_G, test_gene2mod, top_k=1)
    assert hubs.iloc[0]["Gene"] == "hub"
    assert hubs.iloc[0]["Degree"] == 5


def test_node_colors():
    """Test node color assignment."""
    test_G = nx.Graph()
    test_G.add_nodes_from(["A", "B", "C"])
    test_gene2mod = {"A": 0, "B": 0, "C": 1}
    colors = get_node_colors(test_G, test_gene2mod)
    assert len(colors) == 3
    assert colors[0] == colors[1]  # Same module, same color


test_hub_detection()
test_node_colors()
logging.info("All visualization tests passed.")

## 5. Export Results
Save the network figure and hub genes CSV.

In [ ]:
# Recreate and save figure
fig, ax = plt.subplots(figsize=CONFIG.figsize)
nx.draw_networkx_edges(G, pos, alpha=CONFIG.edge_alpha, edge_color="gray", ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, ax=ax)
nx.draw_networkx_labels(
    G, pos, labels=hub_labels, font_size=9, font_weight="bold", ax=ax
)

num_modules = len(set(gene2module.values()))
ax.set_title(
    f"Gene Co-Expression Network — TP53 Associated\n"
    f"{G.number_of_nodes()} genes, {G.number_of_edges()} edges, {num_modules} modules",
    fontsize=14,
)

# Legend
cmap = plt.get_cmap("tab10")
unique_modules = sorted(set(gene2module.values()))
legend_handles = [
    plt.Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=cmap(m % 10),
        markersize=10,
        label=f"Module {m}",
    )
    for m in unique_modules
]
ax.legend(handles=legend_handles, loc="upper left", title="Modules")
ax.axis("off")
plt.tight_layout()

# Save network figure (required deliverable)
network_path = CONFIG.export_dir / f"network_tp53_{CONFIG.handle}.png"
fig.savefig(network_path, dpi=CONFIG.dpi, bbox_inches="tight")
plt.close(fig)
logging.info("[OK] Network visualization saved to: %s", network_path.resolve())

# Save hub genes CSV (required deliverable)
hubs_path = CONFIG.export_dir / f"hubs_tp53_{CONFIG.handle}.csv"
hubs_df.to_csv(hubs_path, index=False)
logging.info("[OK] Hub genes saved to: %s", hubs_path.resolve())

# Additional: hub genes by module
print(f"\nHub genes by module:")
for mod in sorted(hubs_df["Module"].unique()):
    mod_hubs = hubs_df[hubs_df["Module"] == mod]["Gene"].tolist()
    print(f"  Module {mod}: {', '.join(mod_hubs)}")

print(f"\n✓ Task 3 complete: Network visualization and hub genes exported.")